In [73]:
from pyrekordbox import Rekordbox6Database

db = Rekordbox6Database()

In [74]:
# Champs disponibles via SQLAlchemy
tracks = list(db.get_content())
track = tracks[0]

# Les champs utiles pour le schéma
{
    "id":           track.ID,
    "title":        track.Title,
    "artist":       track.Artist.Name if track.Artist else None,
    "album":        track.Album.Name if track.Album else None,
    "label":        track.Label.Name if track.Label else None,
    "bpm":          track.BPM / 100,
    "key":          track.Key.ScaleName if track.Key else None,
    "duration_ms":  track.Length,
    "rating":       track.Rating,
    "color_id":     track.ColorID,
    "play_count":   track.DJPlayCount,
    "isrc":         track.ISRC,
    "file_path":    track.FolderPath,
    "date_added":   track.DateCreated,
    "my_tags":      track.MyTagNames,
    "anlz_path":    track.AnalysisDataPath,
}

{}


{'id': '70654446',
 'title': 'RAMBO',
 'artist': 'Baxter',
 'album': 'RAMBO',
 'label': None,
 'bpm': 144.0,
 'key': '8A',
 'duration_ms': 343,
 'rating': 0,
 'color_id': '0',
 'play_count': 0,
 'isrc': 'QZS672260446',
 'file_path': 'C:/Users/willi/Music/W_MIX/W - Misc. Tracks/Baxter - RAMBO.mp3',
 'date_added': '2026-04-08',
 'my_tags': ['TO_ENERGY', 'TO_RATE', 'TO_CUE', 'TO_LOOP', 'TO_COLOR'],
 'anlz_path': '/PIONEER/USBANLZ/1af/08ede-c0c8-4e7d-a46e-a74d69b8f1f1/ANLZ0000.DAT'}

In [75]:
# Grouper par track pour voir un exemple complet
cues = list(db.get_cue())
print(f"{len(cues)} cues au total")

# Premier track qui a plusieurs cues
by_track = {}
for c in cues:
    by_track.setdefault(c.ContentID, []).append(c)

# Prendre le track avec le plus de cues
sample_id = max(by_track, key=lambda k: len(by_track[k]))
print(f"\nTrack ID {sample_id} — {len(by_track[sample_id])} cues\n")

track = next(t for t in tracks if str(t.ID) == str(sample_id))
print(f"{track.Title} — {track.Artist.Name}")

def cue_label(kind):
    # Kind=0 → memory cue, Kind=1..N → hot cue A, B, C...
    if kind == 0:
        return 'MEM'
    return chr(ord('A') + kind - 1)

for c in sorted(by_track[sample_id], key=lambda x: x.InMsec):
    label = cue_label(c.Kind)
    print(f"  {label} | Kind={c.Kind} | {c.InMsec}ms | Color={c.Color} | Loop={c.ActiveLoop}")

541 cues au total

Track ID 80011739 — 9 cues

Wannabe — VOLAC
  A | Kind=1 | 25ms | Color=-1 | Loop=None
  H | Kind=8 | 15025ms | Color=255 | Loop=0
  B | Kind=2 | 30025ms | Color=-1 | Loop=None
  C | Kind=3 | 75025ms | Color=-1 | Loop=None
  E | Kind=5 | 108775ms | Color=-1 | Loop=None
  F | Kind=6 | 131275ms | Color=-1 | Loop=None
  G | Kind=7 | 157525ms | Color=-1 | Loop=None
  I | Kind=9 | 157525ms | Color=255 | Loop=0
  D | Kind=4 | 157525ms | Color=255 | Loop=0


In [ ]:
tags = list(db.get_my_tag())
tag_songs = list(db.get_my_tag_songs())

# Tags qui ont au moins 1 son
used_tags = {s.MyTag.Name for s in tag_songs}

groups = {t.ID: t.Name for t in tags if t.Attribute == 1}

result = {name: [] for name in groups.values()}
for tag in tags:
    if tag.Attribute == 0 and tag.ParentID in groups:
        # Permet de vérifier si le tag est bien attribué au moins 1 fois :
        if tag.Name in used_tags:
            result[groups[tag.ParentID]].append(tag.Name)

# Supprimer les groupes vides aussi
result = {k: v for k, v in result.items() if v}

print(result)

{'ANALYSIS': ['IMPORTED', 'Deep House'], 'PROCESS': ['TO_SORT', 'TO_ENERGY', 'TO_RATE', 'TO_CUE', 'TO_LOOP', 'TO_COLOR'], 'STYLE': ['Deep House', 'Downtempo', 'French Touch', 'Electro Brut', 'UK House', 'Tech House', 'Melodic Techno', 'Classic/Min. Techno', 'UK Garage', 'Hard/Dark Techno', 'Psytrance', 'Trance Techno', 'Nu-Disco', 'Misc. Tracks'], 'SOURCE': ['SoundCloud', 'Mix', 'Mix_001']}


In [77]:
from collections import defaultdict

tag_songs = list(db.get_my_tag_songs())

# Indexer par nom de tag
by_tag = defaultdict(list)
for s in tag_songs:
    by_tag[s.MyTag.Name].append(s.Content)

# Tous les tracks d'un tag
for track in by_tag['Tech House']:
    print(f"{track.Title} — {track.Artist.Name} | {track.BPM / 100} BPM")

A Levels — GHSTGHSTGHST Sweet Pussy Pauline | 130.0 BPM
Actin' Tough — Dean Turnley | 140.0 BPM
Bad Man Sound — Mosimann Walshy Fire | 132.0 BPM
Bang (The Underground Doesn't Stop) (Original Version) — Laurent Garnier | 128.0 BPM
Bang't — Geeeman | 125.0 BPM
Banggg — O'Flynn | 130.0 BPM
Beat Boy — DJ Cream | 128.0 BPM
Besame — Manda Moor | 130.0 BPM
Beto’s Horns (fred remix) — Fred again.. CA7RIEL & Paco Amoroso | 127.66 BPM
Burnin' (Art Edit) — ANOTR Stef Davidse | 128.0 BPM
Cavaricci (The Carry Nation Remix) — Redance The Carry Nation | 127.0 BPM
Could Heaven Ever Be Like This (Walker & Royce and Chris Lorenzo Remix) — Idris Muhammad Walker & Royce Chris Lorenzo | 127.0 BPM
Dawn (Dub Edit) — z72.52 Justin Shawn Hobbs | 137.0 BPM
Direct Me (Joey Negro Mix) — The Reese Project Joey Negro | 121.82 BPM
Easy Rider — Royal Intention | 128.0 BPM
Es Vedra — Mosimann | 125.0 BPM
Fallin In Love (Butch Remix) — Duck Sauce A-Trak Armand Van Helden Butch | 127.0 BPM
Famosa — Sonora OFFMODE | 128.